## 融合股票池市值数据生成

本 notebook 用于导出“红利低波 x 红利质量”融合股票池历史出现过的股票全集的日频估值/市值数据，供 Qlib/QLab 因子挖掘使用。

导出口径：只要股票历史上进入过融合池，就导出它在配置日期范围内的全部日频市值数据，不按股票池生效区间筛选。


## api 调用方法
get_valuation 获取多个标的在指定交易日范围内的市值表数据

from jqdata import *
get_valuation(security, start_date=None, end_date=None, fields=None, count=None)

获取多个标的在指定交易日范围内的市值表数据

参数
security: 标的code字符串列表或者单个标的字符串
end_date: 查询结束时间
start_date: 查询开始时间，不能与count共用
count: 表示往前查询每一个标的count个交易日的数据，如果期间标的停牌，则该标的返回的市值数据数量小于count
fields: 财务数据中市值表的字段，返回结果中总会包含code、day字段，可用字段如下：
字段	释义
code	股票代码 带后缀.XSHE/.XSHG
day	日期 取数据的日期
capitalization	总股本(万股)
circulating_cap	流通股本(万股)
market_cap	总市值(亿元)
circulating_market_cap	流通市值(亿元)
turnover_ratio	换手率(%)
pe_ratio	市盈率(PE, TTM)
pe_ratio_lyr	市盈率(PE)
pb_ratio	市净率(PB)
ps_ratio	市销率(PS, TTM)
pcf_ratio	市现率(PCF, 现金净流量TTM)
返回值
返回一个dataframe，索引默认是pandas的整数索引，返回的结果中总会包含code、day字段。
注意
每次最多返回5000条数据，更多数据需要根据标的或者时间分多次获取
不要获取当天的估值数据,pe/市值等依赖收盘价的指标是盘后更新的。
示例
from jqdata import *
# 传入单个标的
df1 = get_valuation('000001.XSHE', end_date="2019-11-18", count=3, fields=['capitalization', 'market_cap'])
print(df1)

# 传入多个标的
df2 = get_valuation(['000001.XSHE', '000002.XSHE'], end_date="2019-11-18", count=3, fields=['capitalization', 'market_cap'])
print(df2)


In [ ]:
import os
import gc
import time
import pandas as pd
from jqdata import *
from ipywidgets import IntProgress, HTML, VBox, Layout
from IPython.display import display

# ============================================================
#  融合股票池历史股票全集市值数据导出 — 聚宽研究环境
# ============================================================

# ---- 配置 ----
HYBRID_POOL_SOURCE_FILE = 'all_a_hybrid_hldb_hq_monthly_indcap20_pool_20090101_20260601.csv'
POOL_INPUT_FILE = 'hybrid_finance_pool_input_20090101_20260601.csv'
OUTPUT_FILE = 'hybrid_pool_market_value_20090101_20260531.csv'
START_DATE = '2009-01-01'
END_DATE = '2026-05-31'
BATCH_SIZE = 20
FIELDS = [
    'capitalization', 'circulating_cap', 'market_cap', 'circulating_market_cap',
    'turnover_ratio', 'pe_ratio', 'pe_ratio_lyr', 'pb_ratio', 'ps_ratio', 'pcf_ratio'
]

# ---- 1. 股票池读取与格式转换 ----
def to_jq_code(code):
    code = str(code).strip()
    if code.startswith('sh.'):
        return code.split('.')[1] + '.XSHG'
    if code.startswith('sz.'):
        return code.split('.')[1] + '.XSHE'
    if code.startswith('SH'):
        return code[2:] + '.XSHG'
    if code.startswith('SZ'):
        return code[2:] + '.XSHE'
    if code.endswith('.XSHG') or code.endswith('.XSHE'):
        return code
    raise ValueError('无法识别股票代码格式: {}'.format(code))


def to_qlib_symbol(jq_code):
    jq_code = to_jq_code(jq_code)
    if jq_code.endswith('.XSHG'):
        return 'sh' + jq_code.split('.')[0]
    if jq_code.endswith('.XSHE'):
        return 'sz' + jq_code.split('.')[0]
    return str(jq_code).lower()


def load_hybrid_universe():
    if os.path.exists(POOL_INPUT_FILE):
        pool = pd.read_csv(POOL_INPUT_FILE)
        if 'jq_code' in pool.columns:
            codes = pool['jq_code'].map(to_jq_code)
        elif 'qlib_symbol' in pool.columns:
            codes = pool['qlib_symbol'].map(to_jq_code)
        elif 'instrument' in pool.columns:
            codes = pool['instrument'].map(to_jq_code)
        else:
            raise ValueError('{} 需要包含 jq_code、qlib_symbol 或 instrument 列'.format(POOL_INPUT_FILE))
        return sorted(pd.Series(codes).dropna().unique().tolist())

    if os.path.exists(HYBRID_POOL_SOURCE_FILE):
        pool = pd.read_csv(HYBRID_POOL_SOURCE_FILE)
        if 'instrument' not in pool.columns:
            raise ValueError('{} 缺少 instrument 列'.format(HYBRID_POOL_SOURCE_FILE))
        pool['jq_code'] = pool['instrument'].map(to_jq_code)
        pool['qlib_symbol'] = pool['jq_code'].map(lambda x: to_qlib_symbol(x).upper())
        if {'start_date', 'end_date'}.issubset(pool.columns):
            input_df = pool[['qlib_symbol', 'jq_code', 'start_date', 'end_date']].drop_duplicates()
            input_df.to_csv(POOL_INPUT_FILE, index=False, encoding='utf-8')
            print('已自动生成股票池输入文件:', POOL_INPUT_FILE)
        return sorted(pool['jq_code'].dropna().unique().tolist())

    raise FileNotFoundError(
        '找不到 {} 或 {}。请先运行融合池构建/输入 CSV 生成 cell。'.format(
            POOL_INPUT_FILE, HYBRID_POOL_SOURCE_FILE
        )
    )


jq_codes = load_hybrid_universe()
print('融合池历史股票全集数量:', len(jq_codes))
print('导出日期范围:', START_DATE, '~', END_DATE)

# ---- 2. API 连通性测试 ----
print('正在测试 API 连通性...')
test_df = get_valuation('000001.XSHE', end_date='2020-01-02', count=2, fields=['market_cap'])
print('API 正常，测试返回 {} 行'.format(len(test_df)))
del test_df

# ---- 3. 进度组件 ----
def make_year_ranges(start_date, end_date):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    ranges = []
    for year in range(start.year, end.year + 1):
        ys = max(pd.Timestamp(year=year, month=1, day=1), start)
        ye = min(pd.Timestamp(year=year, month=12, day=31), end)
        ranges.append((ys.strftime('%Y-%m-%d'), ye.strftime('%Y-%m-%d')))
    return ranges


def chunk_list(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]


year_ranges = make_year_ranges(START_DATE, END_DATE)
code_batches = list(chunk_list(jq_codes, BATCH_SIZE))
total_steps = len(code_batches) * len(year_ranges)
bar_w = Layout(width='620px')
p_total = IntProgress(min=0, max=total_steps, layout=bar_w, style={'bar_color': '#1976D2'})
l_title = HTML(value='<h4 style="margin:2px 0">融合池市值数据导出进度</h4>')
l_total = HTML(value='总进度: 准备中...')
l_stats = HTML(value='')
ui = VBox([
    l_title,
    VBox([l_total, p_total]),
    l_stats,
], layout=Layout(padding='10px', border='1px solid #ddd', border_radius='4px'))
display(ui)

# ---- 4. 增量导出 ----
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

pd.DataFrame(columns=['date', 'symbol'] + FIELDS).to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

total_rows = 0
errors = []
step = 0
t0 = time.time()

for batch_idx, batch_codes in enumerate(code_batches, 1):
    symbol_map = {code: to_qlib_symbol(code) for code in batch_codes}
    for ys, ye in year_ranges:
        elapsed = time.time() - t0
        pct = (step + 1) / total_steps * 100
        speed = (step + 1) / elapsed if elapsed > 0 else 0
        eta_s = (total_steps - step - 1) / speed if speed > 0 else 0
        l_total.value = (
            '<b>总进度</b>: {}/{} ({:.1f}%) | 速度 {:.1f} 步/秒 | 剩余 ~{:.1f} 分钟'.format(
                step + 1, total_steps, pct, speed, eta_s / 60
            )
        )
        l_stats.value = (
            '批次 {}/{} | 年份区间 {}~{} | 已写入 <b>{:,}</b> 行 | 错误 {}'.format(
                batch_idx, len(code_batches), ys, ye, total_rows, len(errors)
            )
        )

        try:
            df = get_valuation(batch_codes, start_date=ys, end_date=ye, fields=FIELDS)
            if df is not None and not df.empty:
                df = df.loc[:, ~df.columns.duplicated()].copy()
                df['symbol'] = df['code'].map(symbol_map)
                df['day'] = pd.to_datetime(df['day']).dt.strftime('%Y-%m-%d')
                df = df.rename(columns={'day': 'date'}).drop(columns=['code'])
                df = df[['date', 'symbol'] + FIELDS]
                df.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8')
                total_rows += len(df)
                del df
        except Exception as e:
            if len(errors) < 20:
                errors.append('{}~{} | {}~{} | {}: {}'.format(
                    batch_codes[0], batch_codes[-1], ys, ye, type(e).__name__, e
                ))

        step += 1
        p_total.value = step

    gc.collect()

# ---- 5. 完成与验证 ----
elapsed = time.time() - t0
l_total.value = '<b style="color:green">全部完成!</b>'
l_stats.value = (
    '<b>共 {:,} 行</b> | 股票 {} 只 | 耗时 {:.1f}s ({:.1f}min) | 文件: <code>{}</code> | 错误 {}'.format(
        total_rows, len(jq_codes), elapsed, elapsed / 60, OUTPUT_FILE, len(errors)
    )
)

print('\n导出完成: {:,} 行 -> {}'.format(total_rows, OUTPUT_FILE))
if errors:
    print('错误示例:')
    for err in errors[:5]:
        print(err)

fsize = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print('文件大小: {:.1f} MB'.format(fsize))
preview = pd.read_csv(OUTPUT_FILE, encoding='utf-8')
print('总行数: {:,}'.format(len(preview)))
print('股票数:', preview['symbol'].nunique())
print('日期范围: {} ~ {}'.format(preview['date'].min(), preview['date'].max()))
display(preview.head())
del preview
gc.collect()
